In [1]:
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [3]:
from google.colab import files

uploaded = files.upload()   # choose get_around_pricing_project.csv


Saving get_around_pricing_project.csv to get_around_pricing_project.csv


In [5]:
# 1) Load data
df = pd.read_csv("get_around_pricing_project.csv")
df.head()

,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [6]:
df.describe(include='all')

,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
count,4843.000000,4843,4.843000e+03,4843.00000,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843.000000
unique,NaN,28,NaN,NaN,4,10,8,2,2,2,2,2,2,2,NaN
top,NaN,Citroën,NaN,NaN,diesel,black,estate,True,True,False,False,False,False,True,NaN
freq,NaN,969,NaN,NaN,4641,1633,1606,2662,3839,3865,3881,2613,3674,4514,NaN
mean,2421.000000,NaN,1.409628e+05,128.98823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.214536
std,1398.198007,NaN,6.019674e+04,38.99336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.568268
min,0.000000,NaN,-6.400000e+01,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000
25%,1210.500000,NaN,1.029135e+05,100.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.000000
50%,2421.000000,NaN,1.410800e+05,120.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.000000
75%,3631.500000,NaN,1.751955e+05,135.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.000000


# Data Cleaning

In [7]:
#  Basic data cleaning
# - Remove duplicates
# - Drop rows without target
# - Fill missing categorical with "unknown"
# - Fill missing numerical with median

df = df.drop_duplicates()
df = df.dropna(subset=["rental_price_per_day"])

cat_cols = ["model_key", "fuel", "paint_color", "car_type"]
for col in cat_cols:
    df[col] = df[col].fillna("unknown")

df["mileage"] = df["mileage"].fillna(df["mileage"].median())
df["engine_power"] = df["engine_power"].fillna(df["engine_power"].median())

print("Shape after cleaning:", df.shape)
df.head()


Shape after cleaning: (4843, 15)


,Unnamed: 0,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


# Define Target & Features


In [8]:
# Define target (y) and features (X)
target = "rental_price_per_day"

num_features = ["mileage", "engine_power"]
cat_features = [
    "model_key", "fuel", "paint_color", "car_type",
    "private_parking_available", "has_gps", "has_air_conditioning",
    "automatic_car", "has_getaround_connect", "has_speed_regulator",
    "winter_tires"
]

X = df[num_features + cat_features].copy()
y = df[target].astype(float)


In [9]:
#  Preprocessing + RandomForest model

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Preprocess categorical variables with OneHotEncoder
pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)],
    remainder="passthrough"
)

# Full pipeline: preprocessing + model
model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])


In [13]:
# 🏋️ Train model and evaluate performance
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(Xtrain, ytrain)

y_pred = model.predict(Xtest)
mae = mean_absolute_error(ytest, y_pred)
print("Mean Absolute Error (MAE):", round(mae, 2))


Mean Absolute Error (MAE): 10.61


The error between predicted and actual prices is measured using Mean Absolute Error (MAE).

In this case: MAE = 10.61.

Interpretation: On average, the model’s predicted rental price is off by about 10.6 € per day.

In [14]:
# Save model with input feature list


joblib.dump({"model": model, "input_features": X.columns.tolist()}, "model.joblib")
print("Saved model.joblib")


Saved model.joblib


In [16]:
# ⬇️ Download model to local computer
from google.colab import files
files.download("model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

We evaluated the model with Mean Absolute Error (MAE), which measures the average difference between predicted and actual rental prices.
Our MAE is 10.61 €/day, meaning the model’s predictions are typically off by about 10–11 euros per day compared to the true price.
This shows the model is reasonably accurate for pricing, especially considering the variability in car types, mileage, and options.